In [ ]:
# Load data processed + set parameter

import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler  # bisa ganti MinMaxScaler kalau mau

DATA_PROCESSED_DIR = "../data/processed"

processed_feat_path = os.path.join(
    DATA_PROCESSED_DIR,
    "crypto_1d_2015-01-01_FEATURES_2017-11-09_to_2025-12-19.csv"
)

df = pd.read_csv(processed_feat_path)
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df = df.dropna(subset=["Date"]).sort_values(["Ticker", "Date"]).reset_index(drop=True)

tickers = ["BTC-USD", "ETH-USD", "BNB-USD"]

# Setup windowing 
LOOKBACK = 60                 # input 60 hari terakhir
HORIZONS = [1, 7, 30]         # next-day, weekly, monthly
FEATURE_COLS = ["Close"]      # mulai simpel: pakai Close sebagai fitur
TARGET_COL = "Close"          # target utama: Close

# Split ratio
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15

assert abs((TRAIN_RATIO + VAL_RATIO + TEST_RATIO) - 1.0) < 1e-9
df.head()

,Date,Ticker,Open,High,Low,Close,Volume,LogClose,LogReturn,Return
0,2017-11-09,BNB-USD,2.05314,2.17423,1.89394,1.99077,19192200.0,0.688521,NaN,NaN
1,2017-11-10,BNB-USD,2.00773,2.06947,1.64478,1.79684,11155000.0,0.586030,-0.102492,-0.097415
2,2017-11-11,BNB-USD,1.78628,1.91775,1.61429,1.67047,8178150.0,0.513105,-0.072925,-0.070329
3,2017-11-12,BNB-USD,1.66889,1.67280,1.46256,1.51969,15298700.0,0.418506,-0.094599,-0.090262
4,2017-11-13,BNB-USD,1.52601,1.73502,1.51760,1.68662,12238800.0,0.522727,0.104220,0.109845


In [3]:
# Fungsi windowing (X,y) + metadata tanggal

def make_supervised_windows(dft: pd.DataFrame,
                            lookback: int,
                            horizon: int,
                            feature_cols: list[str],
                            target_col: str):
    """
    Input  : window lookback (t-lookback+1 ... t)
    Target : y = target_col pada t+horizon
    Sample ditandai berdasarkan target_date (tanggal yang diprediksi).
    """
    dft = dft.sort_values("Date").reset_index(drop=True)

    dates = dft["Date"].to_numpy()
    Xraw = dft[feature_cols].to_numpy(dtype=float)
    yraw = dft[target_col].to_numpy(dtype=float)

    X_list, y_list = [], []
    target_dates, last_input_dates = [], []

    n = len(dft)
    # i = index "akhir input window" (t)
    for i in range(lookback - 1, n - horizon):
        x_win = Xraw[i - lookback + 1 : i + 1]      # shape (lookback, n_features)
        y_val = yraw[i + horizon]                   # target di t+horizon

        # skip jika ada NaN (aman)
        if np.isnan(x_win).any() or np.isnan(y_val):
            continue

        X_list.append(x_win)
        y_list.append(y_val)
        last_input_dates.append(dates[i])
        target_dates.append(dates[i + horizon])

    X = np.array(X_list, dtype=float)
    y = np.array(y_list, dtype=float)

    meta = pd.DataFrame({
        "last_input_date": last_input_dates,
        "target_date": target_dates
    })

    return X, y, meta

In [4]:
# Fungsi split time-series (berdasarkan target_date)

def split_by_time(X, y, meta, train_ratio=0.7, val_ratio=0.15):
    """
    Split kronologis berdasarkan urutan meta.target_date.
    """
    meta = meta.reset_index(drop=True)
    n = len(meta)

    train_end = int(n * train_ratio)
    val_end = int(n * (train_ratio + val_ratio))

    idx_train = np.arange(0, train_end)
    idx_val   = np.arange(train_end, val_end)
    idx_test  = np.arange(val_end, n)

    return (
        X[idx_train], y[idx_train], meta.loc[idx_train].reset_index(drop=True),
        X[idx_val],   y[idx_val],   meta.loc[idx_val].reset_index(drop=True),
        X[idx_test],  y[idx_test],  meta.loc[idx_test].reset_index(drop=True)
    )


In [5]:
# Scaling (fit di train) + save dataset per ticker & horizon

OUT_DIR = os.path.join(DATA_PROCESSED_DIR, f"datasets_close_L{LOOKBACK}")
os.makedirs(OUT_DIR, exist_ok=True)

def scale_3d_with_train_scaler(X_train, X_val, X_test):
    """
    Fit scaler pada train saja, lalu transform semua set.
    X shape: (n_samples, lookback, n_features)
    """
    scaler = StandardScaler()

    n_features = X_train.shape[-1]
    Xtr2 = X_train.reshape(-1, n_features)
    Xva2 = X_val.reshape(-1, n_features)
    Xte2 = X_test.reshape(-1, n_features)

    scaler.fit(Xtr2)

    X_train_s = scaler.transform(Xtr2).reshape(X_train.shape)
    X_val_s   = scaler.transform(Xva2).reshape(X_val.shape)
    X_test_s  = scaler.transform(Xte2).reshape(X_test.shape)

    return X_train_s, X_val_s, X_test_s, scaler


for t in tickers:
    dft = df[df["Ticker"] == t].copy()

    # pastikan tidak ada missing close; kalau ada, drop dulu
    dft = dft.dropna(subset=FEATURE_COLS + [TARGET_COL]).reset_index(drop=True)

    for H in HORIZONS:
        X, y, meta = make_supervised_windows(
            dft=dft,
            lookback=LOOKBACK,
            horizon=H,
            feature_cols=FEATURE_COLS,
            target_col=TARGET_COL
        )

        (X_tr, y_tr, m_tr,
        X_va, y_va, m_va,
        X_te, y_te, m_te) = split_by_time(
            X, y, meta,
            train_ratio=TRAIN_RATIO,
            val_ratio=VAL_RATIO
        )

        # scaling pakai train saja
        X_tr_s, X_va_s, X_te_s, scaler = scale_3d_with_train_scaler(X_tr, X_va, X_te)

        # simpan .npz (ringkas & cepat untuk training)
        out_npz = os.path.join(OUT_DIR, f"{t}_H{H}_targetClose.npz")
        np.savez_compressed(
            out_npz,
            X_train=X_tr_s, y_train=y_tr,
            X_val=X_va_s,   y_val=y_va,
            X_test=X_te_s,  y_test=y_te,
            train_last_input_date=m_tr["last_input_date"].astype(str).to_numpy(),
            train_target_date=m_tr["target_date"].astype(str).to_numpy(),
            val_last_input_date=m_va["last_input_date"].astype(str).to_numpy(),
            val_target_date=m_va["target_date"].astype(str).to_numpy(),
            test_last_input_date=m_te["last_input_date"].astype(str).to_numpy(),
            test_target_date=m_te["target_date"].astype(str).to_numpy(),
            feature_cols=np.array(FEATURE_COLS, dtype=object),
            target_col=np.array([TARGET_COL], dtype=object),
            lookback=np.array([LOOKBACK]),
            horizon=np.array([H]),
            scaler_mean=scaler.mean_,
            scaler_scale=scaler.scale_
        )

        print(f"[SAVED] {out_npz}")
        print(f"  X_train: {X_tr_s.shape} | X_val: {X_va_s.shape} | X_test: {X_te_s.shape}")
        print(f"  y_train: {y_tr.shape}   | y_val: {y_va.shape}   | y_test: {y_te.shape}")
        print(f"  date range train targets: {m_tr['target_date'].min().date()} -> {m_tr['target_date'].max().date()}")
        print(f"  date range test  targets: {m_te['target_date'].min().date()} -> {m_te['target_date'].max().date()}")
        print("-" * 70)

[SAVED] ../data/processed\datasets_close_L60\BTC-USD_H1_targetClose.npz
  X_train: (2032, 60, 1) | X_val: (435, 60, 1) | X_test: (436, 60, 1)
  y_train: (2032,)   | y_val: (435,)   | y_test: (436,)
  date range train targets: 2018-01-08 -> 2023-08-01
  date range test  targets: 2024-10-10 -> 2025-12-19
----------------------------------------------------------------------
[SAVED] ../data/processed\datasets_close_L60\BTC-USD_H7_targetClose.npz
  X_train: (2027, 60, 1) | X_val: (435, 60, 1) | X_test: (435, 60, 1)
  y_train: (2027,)   | y_val: (435,)   | y_test: (435,)
  date range train targets: 2018-01-14 -> 2023-08-02
  date range test  targets: 2024-10-11 -> 2025-12-19
----------------------------------------------------------------------
[SAVED] ../data/processed\datasets_close_L60\BTC-USD_H30_targetClose.npz
  X_train: (2011, 60, 1) | X_val: (431, 60, 1) | X_test: (432, 60, 1)
  y_train: (2011,)   | y_val: (431,)   | y_test: (432,)
  date range train targets: 2018-02-06 -> 2023-08-0

In [6]:
# Quick load test (cek satu file .npz)

sample_npz = os.path.join(OUT_DIR, "BTC-USD_H1_targetClose.npz")
d = np.load(sample_npz, allow_pickle=True)

d["X_train"].shape, d["y_train"].shape, d["feature_cols"], d["lookback"], d["horizon"]


((2032, 60, 1),
 (2032,),
 array(['Close'], dtype=object),
 array([60]),
 array([1]))

In [7]:
print("X_train:", d["X_train"].shape, "y_train:", d["y_train"].shape)
print("X_val  :", d["X_val"].shape,   "y_val  :", d["y_val"].shape)
print("X_test :", d["X_test"].shape,  "y_test :", d["y_test"].shape)

print("Train target date range:", d["train_target_date"][0], "->", d["train_target_date"][-1])
print("Val   target date range:", d["val_target_date"][0],   "->", d["val_target_date"][-1])
print("Test  target date range:", d["test_target_date"][0],  "->", d["test_target_date"][-1])

X_train: (2032, 60, 1) y_train: (2032,)
X_val  : (435, 60, 1) y_val  : (435,)
X_test : (436, 60, 1) y_test : (436,)
Train target date range: 2018-01-08 -> 2023-08-01
Val   target date range: 2023-08-02 -> 2024-10-09
Test  target date range: 2024-10-10 -> 2025-12-19


In [8]:
for t in ["BTC-USD","ETH-USD","BNB-USD"]:
    for H in [1,7,30]:
        path = os.path.join(OUT_DIR, f"{t}_H{H}_targetClose.npz")
        d = np.load(path, allow_pickle=True)
        print(t, "H=", H,
            "| X_train", d["X_train"].shape,
            "| target range train", d["train_target_date"][0], "->", d["train_target_date"][-1],
            "| target range test",  d["test_target_date"][0],  "->", d["test_target_date"][-1])


BTC-USD H= 1 | X_train (2032, 60, 1) | target range train 2018-01-08 -> 2023-08-01 | target range test 2024-10-10 -> 2025-12-19
BTC-USD H= 7 | X_train (2027, 60, 1) | target range train 2018-01-14 -> 2023-08-02 | target range test 2024-10-11 -> 2025-12-19
BTC-USD H= 30 | X_train (2011, 60, 1) | target range train 2018-02-06 -> 2023-08-09 | target range test 2024-10-14 -> 2025-12-19
ETH-USD H= 1 | X_train (2032, 60, 1) | target range train 2018-01-08 -> 2023-08-01 | target range test 2024-10-10 -> 2025-12-19
ETH-USD H= 7 | X_train (2027, 60, 1) | target range train 2018-01-14 -> 2023-08-02 | target range test 2024-10-11 -> 2025-12-19
ETH-USD H= 30 | X_train (2011, 60, 1) | target range train 2018-02-06 -> 2023-08-09 | target range test 2024-10-14 -> 2025-12-19
BNB-USD H= 1 | X_train (2032, 60, 1) | target range train 2018-01-08 -> 2023-08-01 | target range test 2024-10-10 -> 2025-12-19
BNB-USD H= 7 | X_train (2027, 60, 1) | target range train 2018-01-14 -> 2023-08-02 | target range test